# Deep Agents - 메트릭 및 비용 추적 미들웨어

## 개요

이 노트북에서는 Deep Agents에 메트릭 및 비용 추적 기능을 추가하는 커스텀 미들웨어를 구현합니다.
`AsyncCallbackHandler`를 활용하여 LLM 호출, 토큰 사용량, 비용을 실시간으로 추적합니다.

## Deep Agents + Callbacks 통합

Deep Agents는 LangChain의 콜백 시스템과 완벽하게 통합됩니다:
- `callbacks` 파라미터로 핸들러 등록
- LLM 라이프사이클 이벤트 자동 추적
- 미들웨어와 콜백의 협력

## 학습 목표

- Deep Agent에 콜백 핸들러 통합
- LLM 메트릭 실시간 추적
- 모델별 비용 계산
- 예산 관리 시스템 구현

## 환경 설정

In [ ]:
# 필요한 라이브러리 설치
# !pip install langchain langchain-anthropic langgraph deepagents

In [ ]:
import os
from datetime import datetime
from typing import Dict, List, Any, Optional
from collections import defaultdict
import asyncio
import time

# Deep Agents
from deepagents import create_deep_agent
from deepagents.middleware import TodoListMiddleware

# Callbacks
from langchain.callbacks.base import AsyncCallbackHandler, BaseCallbackHandler
from langchain.schema import LLMResult, AgentAction, AgentFinish

# Tools
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langchain_anthropic import ChatAnthropic

# API 키 설정
# os.environ["ANTHROPIC_API_KEY"] = "your-api-key"

print("✅ Deep Agents 및 Callbacks 라이브러리 import 완료")

## 1. MetricsMiddleware 구현

`AsyncCallbackHandler`를 상속하여 LLM 호출과 토큰 사용량을 실시간으로 추적합니다.

In [ ]:
class MetricsMiddleware(AsyncCallbackHandler):
    """Deep Agent용 메트릭 추적 미들웨어
    
    AsyncCallbackHandler를 상속받아 LLM 라이프사이클 이벤트를 추적합니다.
    Deep Agent의 callbacks 파라미터에 등록하여 사용합니다.
    """
    
    def __init__(self, verbose: bool = True):
        super().__init__()
        self.verbose = verbose
        self.reset()
    
    def reset(self):
        """메트릭 초기화"""
        self.requests = 0  # LLM 호출 횟수
        self.tokens = 0  # 총 토큰 수
        self.input_tokens = 0  # 입력 토큰
        self.output_tokens = 0  # 출력 토큰
        self.tool_calls = 0  # 도구 호출 횟수
        self.tool_usage = defaultdict(int)  # 도구별 사용 횟수
        self.errors = 0  # 오류 횟수
        self.start_time = datetime.now()
        self.llm_durations = []  # LLM 호출 소요 시간들
        self.current_llm_start = None
    
    async def on_llm_start(
        self, serialized: Dict[str, Any], prompts: List[str], **kwargs
    ) -> None:
        """LLM 호출 시작 시 호출됨"""
        self.requests += 1
        self.current_llm_start = time.time()
        
        if self.verbose:
            print(f"\n🤖 [Metrics] LLM 요청 #{self.requests} 시작")
    
    async def on_llm_new_token(self, token: str, **kwargs) -> None:
        """새 토큰이 생성될 때마다 호출됨 (스트리밍 모드)"""
        self.tokens += 1
        self.output_tokens += 1
    
    async def on_llm_end(self, response: LLMResult, **kwargs) -> None:
        """LLM 호출 완료 시 호출됨"""
        # 소요 시간 계산
        if self.current_llm_start:
            duration = time.time() - self.current_llm_start
            self.llm_durations.append(duration)
        
        # 토큰 사용량 업데이트
        if response.llm_output:
            token_usage = response.llm_output.get('token_usage', {})
            if token_usage:
                self.input_tokens += token_usage.get('prompt_tokens', 0)
                output = token_usage.get('completion_tokens', 0)
                self.output_tokens += output
                self.tokens = self.input_tokens + self.output_tokens
        
        if self.verbose:
            print(f"✅ [Metrics] LLM 요청 완료 (소요: {duration:.2f}초, 총 토큰: {self.tokens})")
    
    async def on_llm_error(self, error: Exception, **kwargs) -> None:
        """LLM 오류 발생 시 호출됨"""
        self.errors += 1
        if self.verbose:
            print(f"❌ [Metrics] LLM 오류 발생: {error}")
    
    async def on_tool_start(
        self, serialized: Dict[str, Any], input_str: str, **kwargs
    ) -> None:
        """도구 호출 시작 시 호출됨"""
        self.tool_calls += 1
        tool_name = serialized.get('name', 'unknown')
        self.tool_usage[tool_name] += 1
        
        if self.verbose:
            print(f"🔧 [Metrics] 도구 호출: {tool_name} (총 {self.tool_calls}회)")
    
    async def on_tool_error(self, error: Exception, **kwargs) -> None:
        """도구 오류 발생 시 호출됨"""
        self.errors += 1
    
    async def on_agent_finish(self, finish: AgentFinish, **kwargs) -> None:
        """에이전트 작업 완료 시 호출됨"""
        if self.verbose:
            print(f"\n{'='*70}")
            print(f"[Metrics] Requests = {self.requests}, Tokens = {self.tokens}")
            print(f"{'='*70}")
            self._print_summary()
    
    def _print_summary(self):
        """메트릭 요약 출력"""
        elapsed = (datetime.now() - self.start_time).total_seconds()
        avg_duration = sum(self.llm_durations) / len(self.llm_durations) if self.llm_durations else 0
        
        print(f"\n📊 메트릭 요약:")
        print(f"   ⏱️  총 소요 시간: {elapsed:.2f}초")
        print(f"   🤖 LLM 호출 횟수: {self.requests}회")
        print(f"   📝 총 토큰 사용량: {self.tokens:,}개")
        print(f"      - 입력 토큰: {self.input_tokens:,}개")
        print(f"      - 출력 토큰: {self.output_tokens:,}개")
        print(f"   🔧 도구 호출 횟수: {self.tool_calls}회")
        
        if self.tool_usage:
            print(f"\n   도구별 사용량:")
            for tool_name, count in sorted(self.tool_usage.items(), key=lambda x: x[1], reverse=True):
                print(f"      - {tool_name}: {count}회")
        
        if self.llm_durations:
            print(f"\n   ⚡ LLM 평균 응답 시간: {avg_duration:.2f}초")
        
        if self.errors > 0:
            print(f"   ❌ 오류 횟수: {self.errors}회")
    
    def get_metrics(self) -> Dict[str, Any]:
        """메트릭 데이터 반환"""
        elapsed = (datetime.now() - self.start_time).total_seconds()
        avg_duration = sum(self.llm_durations) / len(self.llm_durations) if self.llm_durations else 0
        
        return {
            'elapsed_time': elapsed,
            'llm_requests': self.requests,
            'total_tokens': self.tokens,
            'input_tokens': self.input_tokens,
            'output_tokens': self.output_tokens,
            'tool_calls': self.tool_calls,
            'tool_usage': dict(self.tool_usage),
            'errors': self.errors,
            'avg_llm_duration': avg_duration,
        }


print("✅ MetricsMiddleware 정의 완료")

## 2. 비용 추적 핸들러

토큰 사용량을 기반으로 실시간 비용을 계산하고 예산을 관리합니다.

In [ ]:
class CostTrackingHandler(AsyncCallbackHandler):
    """Deep Agent용 비용 추적 핸들러
    
    모델별 토큰 단가를 설정하고 실시간으로 비용을 계산합니다.
    예산을 설정하여 초과 시 경고를 발생시킬 수 있습니다.
    """
    
    # 모델별 비용 (1M 토큰당 USD)
    MODEL_COSTS = {
        "claude-sonnet-4-5-20250929": {
            "input": 3.0,
            "output": 15.0,
            "name": "Claude Sonnet 4.5"
        },
        "claude-3-5-sonnet-20241022": {
            "input": 3.0,
            "output": 15.0,
            "name": "Claude 3.5 Sonnet"
        },
        "gpt-4o": {
            "input": 5.0,
            "output": 15.0,
            "name": "GPT-4o"
        },
        "gpt-4o-mini": {
            "input": 0.15,
            "output": 0.6,
            "name": "GPT-4o Mini"
        },
    }
    
    def __init__(
        self, 
        model_name: str = "claude-sonnet-4-5-20250929", 
        budget: Optional[float] = None, 
        verbose: bool = True
    ):
        super().__init__()
        self.model_name = model_name
        self.budget = budget  # USD
        self.verbose = verbose
        self.reset()
    
    def reset(self):
        """비용 추적 초기화"""
        self.input_tokens = 0
        self.output_tokens = 0
        self.total_cost = 0.0
        self.input_cost = 0.0
        self.output_cost = 0.0
        self.cost_history = []
        self.budget_warnings = 0
    
    def _calculate_cost(self, input_tokens: int, output_tokens: int) -> Dict[str, float]:
        """토큰 수를 기반으로 비용 계산"""
        if self.model_name not in self.MODEL_COSTS:
            return {"input_cost": 0.0, "output_cost": 0.0, "total_cost": 0.0}
        
        costs = self.MODEL_COSTS[self.model_name]
        input_cost = (input_tokens / 1_000_000) * costs["input"]
        output_cost = (output_tokens / 1_000_000) * costs["output"]
        
        return {
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": input_cost + output_cost
        }
    
    async def on_llm_end(self, response: LLMResult, **kwargs) -> None:
        """LLM 호출 완료 시 비용 계산"""
        if response.llm_output:
            token_usage = response.llm_output.get('token_usage', {})
            if token_usage:
                call_input_tokens = token_usage.get('prompt_tokens', 0)
                call_output_tokens = token_usage.get('completion_tokens', 0)
                
                # 비용 계산
                costs = self._calculate_cost(call_input_tokens, call_output_tokens)
                
                # 누적
                self.input_tokens += call_input_tokens
                self.output_tokens += call_output_tokens
                self.input_cost += costs['input_cost']
                self.output_cost += costs['output_cost']
                self.total_cost += costs['total_cost']
                
                # 이력 저장
                self.cost_history.append({
                    'timestamp': datetime.now().isoformat(),
                    'input_tokens': call_input_tokens,
                    'output_tokens': call_output_tokens,
                    'cost': costs['total_cost']
                })
                
                # 예산 확인
                if self.budget and self.total_cost > self.budget:
                    self.budget_warnings += 1
                    if self.verbose:
                        print(f"\n⚠️  [Budget Alert] 예산 초과!")
                        print(f"   설정 예산: ${self.budget:.6f}")
                        print(f"   현재 비용: ${self.total_cost:.6f}")
                        print(f"   초과 금액: ${self.total_cost - self.budget:.6f}\n")
                
                if self.verbose:
                    print(f"💰 [Cost] 이번 호출: ${costs['total_cost']:.6f} | 누적: ${self.total_cost:.6f}")
                    if self.budget:
                        remaining = self.budget - self.total_cost
                        percent = (self.total_cost / self.budget) * 100
                        print(f"   예산 사용: {percent:.1f}% | 남은 예산: ${remaining:.6f}")
    
    def get_cost_report(self) -> str:
        """비용 리포트 생성"""
        model_info = self.MODEL_COSTS.get(self.model_name, {})
        model_display_name = model_info.get('name', self.model_name)
        
        report = f"""\n{'='*70}
💰 비용 리포트
{'='*70}
🤖 모델: {model_display_name}

📊 토큰 사용량:
   입력 토큰:  {self.input_tokens:,}개
   출력 토큰:  {self.output_tokens:,}개
   전체 토큰:  {self.input_tokens + self.output_tokens:,}개

💵 비용 상세:
   입력 비용:  ${self.input_cost:.6f}
   출력 비용:  ${self.output_cost:.6f}
   총 비용:    ${self.total_cost:.6f}
"""
        
        if self.budget:
            remaining = self.budget - self.total_cost
            usage_percent = (self.total_cost / self.budget) * 100
            report += f"""\n📊 예산 현황:
   설정 예산:  ${self.budget:.6f}
   사용 금액:  ${self.total_cost:.6f} ({usage_percent:.1f}%)
   남은 예산:  ${remaining:.6f}
"""
            if self.budget_warnings > 0:
                report += f"   ⚠️  예산 초과 경고: {self.budget_warnings}회\n"
        
        report += "=" * 70
        return report


print("✅ CostTrackingHandler 정의 완료")

## 3. Deep Agent with Metrics & Cost Tracking

이제 메트릭과 비용 추적이 통합된 Deep Agent를 생성합니다.

In [ ]:
# 작업 도구 정의
@tool
def search_database(query: str) -> str:
    """데이터베이스를 검색합니다.
    
    Args:
        query: 검색 쿼리
    """
    time.sleep(0.5)
    return f"검색 결과: '{query}'에 대한 8개의 레코드를 찾았습니다."


@tool
def analyze_data(data_id: str) -> str:
    """데이터를 분석합니다.
    
    Args:
        data_id: 분석할 데이터 ID
    """
    time.sleep(0.7)
    return f"데이터 {data_id} 분석 완료: 주요 인사이트 3개 발견, 신뢰도 92%"


@tool
def generate_report(topic: str, format: str = "PDF") -> str:
    """리포트를 생성합니다.
    
    Args:
        topic: 리포트 주제
        format: 리포트 형식 (PDF, HTML 등)
    """
    time.sleep(1.0)
    return f"'{topic}' 리포트 생성 완료 ({format} 형식, 12페이지, 5개 차트 포함)"


print("✅ 작업 도구 정의 완료")

In [ ]:
# 핸들러 생성
metrics_handler = MetricsMiddleware(verbose=True)
cost_handler = CostTrackingHandler(
    model_name="claude-sonnet-4-5-20250929",
    budget=1.0,  # $1.00 예산
    verbose=True
)

# Deep Agent 생성 (callbacks로 핸들러 등록)
agent = create_deep_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_database, analyze_data, generate_report],
    callbacks=[metrics_handler, cost_handler],  # 핸들러 등록
    middleware=[
        TodoListMiddleware(
            system_prompt="""작업 수행 시:
1. write_todos로 계획을 수립하세요
2. 각 작업을 실행하세요
3. 완료 시 상태를 업데이트하세요
"""
        )
    ]
)

print("✅ 메트릭 및 비용 추적이 통합된 Deep Agent 생성 완료")
print("\n등록된 핸들러:")
print("  - MetricsMiddleware: LLM 호출, 토큰, 도구 사용 추적")
print("  - CostTrackingHandler: 실시간 비용 계산 및 예산 관리")

## 4. Deep Agent 실행 시뮬레이션

실제 작업을 시뮬레이션하면서 메트릭과 비용을 추적합니다.

In [ ]:
def simulate_deep_agent_with_tracking():
    """메트릭 및 비용 추적 시뮬레이션"""
    
    print("🚀 Deep Agent 작업 시작\n")
    print("작업: 고객 데이터 분석 및 리포트 생성")
    print(f"예산: ${cost_handler.budget}\n")
    
    # LLM 호출 시뮬레이션
    from langchain.schema import LLMResult
    
    async def simulate_llm_call(input_tokens: int, output_tokens: int, task_name: str):
        """LLM 호출 시뮬레이션"""
        print(f"\n{'='*70}")
        print(f"🔄 {task_name}")
        print(f"{'='*70}")
        
        # LLM 시작
        await metrics_handler.on_llm_start({}, ["prompt"])
        
        # 처리 시간 시뮬레이션
        await asyncio.sleep(0.5)
        
        # LLM 완료
        mock_response = LLMResult(
            generations=[],
            llm_output={
                'token_usage': {
                    'prompt_tokens': input_tokens,
                    'completion_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens
                }
            }
        )
        await metrics_handler.on_llm_end(mock_response)
        await cost_handler.on_llm_end(mock_response)
    
    # 작업 1: 계획 수립
    asyncio.run(simulate_llm_call(100, 150, "작업 계획 수립"))
    
    # 도구 호출 1: 데이터베이스 검색
    asyncio.run(metrics_handler.on_tool_start({"name": "search_database"}, "고객 데이터"))
    result = search_database.invoke({"query": "2024년 고객 거래 데이터"})
    print(f"\n{result}")
    
    # 작업 2: 검색 결과 분석
    asyncio.run(simulate_llm_call(200, 250, "검색 결과 분석 및 다음 단계 결정"))
    
    # 도구 호출 2-4: 데이터 분석 (여러 번)
    for i in range(1, 4):
        asyncio.run(metrics_handler.on_tool_start({"name": "analyze_data"}, f"data_{i}"))
        result = analyze_data.invoke({"data_id": f"customer_data_{i}"})
        print(f"\n{result}")
    
    # 작업 3: 분석 결과 종합
    asyncio.run(simulate_llm_call(300, 400, "데이터 분석 결과 종합"))
    
    # 도구 호출 5: 리포트 생성
    asyncio.run(metrics_handler.on_tool_start({"name": "generate_report"}, "고객 분석 리포트"))
    result = generate_report.invoke({"topic": "2024 고객 거래 분석", "format": "PDF"})
    print(f"\n{result}")
    
    # 작업 4: 최종 검토
    asyncio.run(simulate_llm_call(150, 100, "리포트 최종 검토 및 요약"))
    
    # 에이전트 완료
    asyncio.run(metrics_handler.on_agent_finish(
        AgentFinish(return_values={"output": "완료"}, log="작업 완료")
    ))
    
    print("\n" + "="*70)
    print("✅ 모든 작업 완료")
    print("="*70)
    
    # 비용 리포트
    print(cost_handler.get_cost_report())


# 시뮬레이션 실행
simulate_deep_agent_with_tracking()

## 5. 실제 Deep Agent 사용 예제

실제로 LLM과 함께 Deep Agent를 실행하는 예제입니다.
(주석 처리됨 - API 키가 있을 때 실행)

In [ ]:
# 실제 Deep Agent 실행 예제 (API 키 필요)
"""
# API 키 설정
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# 핸들러 초기화
metrics_handler.reset()
cost_handler.reset()

# Deep Agent 실행
result = agent.invoke({
    "messages": [HumanMessage(content="""
다음 작업을 수행하고, 각 단계별로 진행 상태를 보고해주세요:

1. 'AI 시장 동향 2024'를 데이터베이스에서 검색하세요
2. 검색된 데이터 3개를 상세히 분석하세요
3. 분석 결과를 바탕으로 종합 리포트를 생성하세요

비용과 성능을 모니터링하면서 작업을 진행해주세요.
""")]
})

print("\n" + "="*70)
print("최종 결과")
print("="*70)
print(result['messages'][-1].content)

# 최종 리포트
print(cost_handler.get_cost_report())

# 메트릭 요약
metrics = metrics_handler.get_metrics()
print(f"\n전체 메트릭:")
print(f"  - LLM 호출: {metrics['llm_requests']}회")
print(f"  - 토큰 사용: {metrics['total_tokens']:,}개")
print(f"  - 도구 호출: {metrics['tool_calls']}회")
print(f"  - 소요 시간: {metrics['elapsed_time']:.2f}초")
"""

print("💡 실제 실행하려면 위 코드의 주석을 해제하고 API 키를 설정하세요.")

## 6. 비용 최적화 분석

수집된 메트릭을 분석하여 비용 최적화 방안을 제시합니다.

In [ ]:
def analyze_cost_optimization():
    """비용 최적화 분석"""
    
    metrics = metrics_handler.get_metrics()
    
    print("\n" + "="*70)
    print("🔍 비용 최적화 분석")
    print("="*70)
    
    # 1. 토큰 효율성
    total_tokens = metrics['total_tokens']
    requests = metrics['llm_requests']
    avg_tokens = total_tokens / requests if requests > 0 else 0
    
    print(f"\n📊 토큰 효율성:")
    print(f"   평균 토큰/요청: {avg_tokens:.0f}개")
    print(f"   입력:출력 비율: {metrics['input_tokens']}:{metrics['output_tokens']}")
    
    if metrics['input_tokens'] > metrics['output_tokens'] * 2:
        print("\n💡 최적화 제안:")
        print("   입력 토큰이 출력의 2배 이상입니다.")
        print("   - 프롬프트를 더 간결하게 작성하세요")
        print("   - 불필요한 컨텍스트를 제거하세요")
    
    # 2. 모델 비교
    current_cost = cost_handler.total_cost
    
    print(f"\n💰 모델 비교:")
    print(f"   현재 모델: {cost_handler.model_name}")
    print(f"   현재 비용: ${current_cost:.6f}")
    
    # 다른 모델 사용 시 예상 비용
    for model_name, model_info in CostTrackingHandler.MODEL_COSTS.items():
        if model_name != cost_handler.model_name:
            input_cost = (metrics['input_tokens'] / 1_000_000) * model_info['input']
            output_cost = (metrics['output_tokens'] / 1_000_000) * model_info['output']
            total = input_cost + output_cost
            savings = ((current_cost - total) / current_cost * 100) if current_cost > 0 else 0
            
            if savings > 0:
                print(f"\n   {model_info['name']} 사용 시:")
                print(f"     예상 비용: ${total:.6f}")
                print(f"     절감 가능: {savings:.1f}%")
    
    # 3. 도구 사용 패턴
    if metrics['tool_usage']:
        print(f"\n🔧 도구 사용 패턴:")
        most_used = max(metrics['tool_usage'].items(), key=lambda x: x[1])
        print(f"   가장 많이 사용된 도구: {most_used[0]} ({most_used[1]}회)")
        
        if metrics['tool_calls'] > requests * 2:
            print("\n💡 최적화 제안:")
            print("   도구 호출이 LLM 요청의 2배 이상입니다.")
            print("   - 배치 처리를 고려하세요")
            print("   - 캐싱을 활용하세요")
    
    print("\n" + "="*70)


# 최적화 분석 실행
analyze_cost_optimization()

## 7. 모델 비교 도구

동일한 작업을 다른 모델로 수행했을 때의 비용을 비교합니다.

In [ ]:
def compare_model_costs():
    """모델별 비용 비교"""
    
    metrics = metrics_handler.get_metrics()
    input_tokens = metrics['input_tokens']
    output_tokens = metrics['output_tokens']
    
    print("\n" + "="*70)
    print("💰 모델별 비용 비교")
    print("="*70)
    print(f"\n토큰 사용량: 입력 {input_tokens:,}개, 출력 {output_tokens:,}개\n")
    
    results = []
    
    for model_name, model_info in CostTrackingHandler.MODEL_COSTS.items():
        input_cost = (input_tokens / 1_000_000) * model_info['input']
        output_cost = (output_tokens / 1_000_000) * model_info['output']
        total_cost = input_cost + output_cost
        
        results.append({
            'name': model_info['name'],
            'total_cost': total_cost
        })
    
    # 비용 순으로 정렬
    results.sort(key=lambda x: x['total_cost'])
    
    # 결과 출력
    for i, result in enumerate(results, 1):
        icon = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
        print(f"{icon}{i}. {result['name']:<25} ${result['total_cost']:.6f}")
    
    # 절감 가능 금액
    cheapest = results[0]
    most_expensive = results[-1]
    savings = most_expensive['total_cost'] - cheapest['total_cost']
    savings_percent = (savings / most_expensive['total_cost']) * 100
    
    print(f"\n최대 절감 가능: ${savings:.6f} ({savings_percent:.1f}%)")
    print("="*70)


# 모델 비교 실행
compare_model_costs()

---

## 핵심 요약

### Deep Agents + Callbacks 통합

1. **간단한 통합**: `callbacks` 파라미터로 핸들러 등록
2. **자동 추적**: LLM 라이프사이클 이벤트 자동 발생
3. **미들웨어와 협력**: TodoList 등 다른 미들웨어와 함께 작동

### MetricsMiddleware 핵심

```python
class MetricsMiddleware(AsyncCallbackHandler):
    async def on_llm_start(self, **kwargs):
        self.requests += 1
    
    async def on_llm_new_token(self, token: str, **kwargs):
        self.tokens += 1
    
    async def on_agent_finish(self, **kwargs):
        print(f"[Metrics] Requests = {self.requests}, Tokens = {self.tokens}")
```

### CostTrackingHandler 핵심

1. **모델별 단가 설정**: MODEL_COSTS 딕셔너리
2. **실시간 비용 계산**: on_llm_end에서 자동 계산
3. **예산 관리**: 초과 시 자동 경고

### 실무 적용

- **비용 모니터링**: 실시간으로 비용 추적 및 예산 관리
- **성능 최적화**: 메트릭 분석으로 병목 지점 파악
- **모델 선택**: 비용 대비 성능이 좋은 모델 선택
- **투명성**: 명확한 비용 산출 및 보고

### 참고 자료

- [Deep Agents Middleware 문서](https://docs.langchain.com/oss/python/deepagents/middleware)
- [LangChain Callbacks 문서](https://python.langchain.com/docs/modules/callbacks/)
- [Anthropic Pricing](https://www.anthropic.com/pricing)
- [OpenAI Pricing](https://openai.com/pricing)